# Capítulo 3 · Algoritmo de Bernstein-Vazirani

## Objetivos

1. Comprender el problema de Bernstein-Vazirani como extensión del algoritmo de Deutsch.
2. Implementar el oráculo lineal y el algoritmo completo.
3. Verificar que se recupera la cadena secreta $\mathbf{s}$ en una sola consulta.

---

## 3B.1 El problema

Dada la función $f(\mathbf{x}) = \mathbf{s} \cdot \mathbf{x} \pmod{2}$ (producto escalar módulo 2 con una cadena secreta $\mathbf{s} \in \{0,1\}^n$), Bernstein-Vazirani determina $\mathbf{s}$ con una sola llamada al oráculo, mientras que un algoritmo clásico necesita $n$ consultas.

El circuito es idéntico al de Deutsch-Jozsa pero el oráculo codifica el producto escalar:

$$U_f|\mathbf{x}\rangle|-\rangle = (-1)^{\mathbf{s}\cdot\mathbf{x}}|\mathbf{x}\rangle|-\rangle$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

In [ ]:
def bv_oracle(secret: str) -> QuantumCircuit:
    """Oráculo para el algoritmo de Bernstein-Vazirani.

    El oráculo implementa f(x) = s·x mod 2, donde s es la cadena secreta.
    Aplica CNOT desde el qubit i al qubit ancilla cuando s[i] = '1'.

    Parámetros
    ----------
    secret : str
        Cadena binaria secreta de longitud n.
    """
    n = len(secret)
    qc = QuantumCircuit(n + 1, name=f'BV_Oracle({secret})')
    for i, bit in enumerate(reversed(secret)):
        if bit == '1':
            qc.cx(i, n)
    return qc


def bernstein_vazirani(secret: str) -> QuantumCircuit:
    """Circuito completo del algoritmo de Bernstein-Vazirani."""
    n = len(secret)
    qc = QuantumCircuit(n + 1, n)

    # Ancilla en |1〉
    qc.x(n)
    qc.barrier()

    # Hadamard en todos los qubits
    qc.h(range(n + 1))
    qc.barrier()

    # Oráculo
    oracle = bv_oracle(secret)
    qc.compose(oracle, inplace=True)
    qc.barrier()

    # Hadamard sobre qubits de entrada
    qc.h(range(n))
    qc.barrier()

    # Medida
    qc.measure(range(n), range(n))
    return qc


# Prueba con distintas cadenas secretas
backend = AerSimulator()
test_secrets = ['10110', '11111', '10000', '01010', '110110']

print('Verificación del algoritmo de Bernstein-Vazirani:')
print(f'{"Secreto":<15} {"Recuperado":<15} {"Correcto"}')
print('-' * 40)

for secret in test_secrets:
    qc = bernstein_vazirani(secret)
    job = backend.run(qc, shots=1)
    counts = job.result().get_counts()
    # El resultado está invertido (convenio Qiskit)
    recovered = list(counts.keys())[0][::-1]
    correct = (recovered == secret)
    print(f'{secret:<15} {recovered:<15} {"✓" if correct else "✗"}')

In [ ]:
# Visualización del circuito para un ejemplo corto
secret_demo = '10110'
qc_demo = bernstein_vazirani(secret_demo)
print(f'Circuito Bernstein-Vazirani para s = {secret_demo}:')
print(qc_demo.draw('text'))

## 3B.2 Ejercicios propuestos

1. Demuestra formalmente (siguiendo la evolución de estados) que el estado antes de la medida es exactamente $|\mathbf{s}\rangle|{-}\rangle$.

2. ¿Cuántas llamadas al oráculo necesita un algoritmo clásico determinista? ¿Y uno probabilístico que acepta un error del 5%?

3. Genera una cadena secreta aleatoria de longitud 20 y verifica que el algoritmo la recupera en una sola medición.